# 10. Custom middleware from need to a working agent

We will build one layer that limits the number of model calls in a conversation. This is a small, complete example: requirement → hook → connection → result → verification. The model is scripted, so the lesson does not require an API key or a running Ollama instance. This model **does not make decisions**: it returns prepared responses.

[Official custom middleware hooks](https://docs.langchain.com/oss/python/langchain/middleware/custom)

In [ ]:
# All lesson imports are here.
from typing import NotRequired

from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import InMemorySaver


## 1. Choosing a starting point

The agent can call the model multiple times. `before_model(state, runtime)` runs before each such attempt. We see the state of the conversation and can return a **partial update**. When the budget is exhausted, we will direct the graph to the end. `wrap_model_call(request, handler)` wraps the actual model call; it must return the result of `handler(request)` unless it intentionally replaces the response.

`before_agent` / `after_agent` run once per agent call. `before_model` / `after_model` run around each model turn. Model and tool wrappers work around individual calls. You do not need to implement all hooks.

## 2. We design the state and contract

`model_calls` belongs to the state because it needs to increase within the scope of the conversation. `AgentState` already contains `messages`. We return only the changed fields; we do not edit the `state` dictionary in place. Thanks to the checkpointer, the counter will pass to the next `invoke` with the same `thread_id`. The limit in the constructor is a configuration of the middleware itself.

In [ ]:
class BudgetState(AgentState):
    model_calls: NotRequired[int]


class ModelBudget(AgentMiddleware):
    state_schema = BudgetState

    def __init__(self, max_calls: int):
        if max_calls < 1:
            raise ValueError("max_calls must be positive")
        self.max_calls = max_calls
        self.events = []  # Only for observation in this lesson, not part of the conversation state.

    @hook_config(can_jump_to=["end"])
    def before_model(self, state: BudgetState, runtime):
        used = state.get("model_calls", 0)
        if used >= self.max_calls:
            # This message is added to the state, and the jump ends the execution without calling the model.
            self.events.append("blocked")
            return {"messages": [AIMessage(content="Model call limit reached.")], "jump_to": "end"}
        # The returned state fragment will be merged by the graph before the model call.
        return {"model_calls": used + 1}

    def wrap_model_call(self, request, handler):
        self.events.append("model:enter")
        try:
            return handler(request)  # Single call; we do not repeat it by mistake.
        finally:
            self.events.append("model:leave")


## 3. Connecting to `create_agent`

The `middleware=[...]` list tells the agent when to call the hooks. The checkpointer saves `model_calls` along with the history. The scripted model has exactly two responses. `events` shows whether the model call actually took place.

In [ ]:
budget = ModelBudget(max_calls=2)
model = FakeMessagesListChatModel(responses=[
    AIMessage(content="First response."),
    AIMessage(content="Second response."),
])
agent = create_agent(model=model, middleware=[budget], checkpointer=InMemorySaver())
thread = {"configurable": {"thread_id": "budget-demo"}}


In [ ]:
first = agent.invoke({"messages": [("user", "Question 1")]}, config=thread)
first["model_calls"], first["messages"][-1].content, budget.events


In [ ]:
second = agent.invoke({"messages": [("user", "Question 2")]}, config=thread)
second["model_calls"], second["messages"][-1].content, budget.events


In [ ]:
third = agent.invoke({"messages": [("user", "Question 3")]}, config=thread)
third["model_calls"], third["messages"][-1].content, budget.events


The third call ends in `before_model`; it should not add `model:enter`. Let's check this without guessing from the text response alone.

In [ ]:
assert (first["model_calls"], second["model_calls"], third["model_calls"]) == (1, 2, 2)
assert budget.events == ["model:enter", "model:leave", "model:enter", "model:leave", "blocked"]
assert third["messages"][-1].content == "Model call limit reached."


## 4. New conversation, new budget

The middleware instance is shared, so `self.events` accumulates observations from **both** conversations. This field must not be used as a permission counter or a limit for a single user. The correct counter is in `state`, separated by `thread_id`.

In [ ]:
new_thread = {"configurable": {"thread_id": "budget-new"}}
new_result = agent.invoke({"messages": [("user", "New conversation")]}, config=new_thread)
new_result["model_calls"], new_result["messages"][-1].content


## 5. Recipe for custom middleware

1. Name one requirement and choose the hook closest to the event (`before_model` for pre-model updates, `wrap_tool_call` for controlling a specific tool).
2. Determine which data belongs to the state, which to `runtime.context`, and which to the layer configuration. Data from a single conversation should not reside in `self`.
3. Define a minimal state schema only when adding fields. A node hook returns a partial update or `None`; a wrapper returns the handler's response or an explicit replacement.
4. Connect the middleware to `create_agent`, run a small reproducible example, and check the effect and the number of actual calls.
5. Regarding errors and retries, remember side effects and idempotency. For `ainvoke`, appropriate asynchronous hooks are required.

Further, more complex cases (tools, order of multiple layers, retry, async, streaming) are in [workshop 16](06_middlewares/16_custom_middleware.ipynb).